# 📓 YOLO Explainable AI (XAI) - Jupyter Notebook (Week 1)

สมุดโน้ตนี้สร้างขึ้นสำหรับการศึกษาและทดลองรันอัลกอริทึม **Explainable AI (XAI)** บนโมเดลตรวจจับวัตถุตระกูล **YOLO** (ในที่นี้คือ YOLOv8) เพื่อให้เห็นและเปรียบเทียบว่าเลเยอร์ภายในของ Neural Network มองภาพและตัดสินใจวิเคราะห์อย่างไร

### สิ่งที่จะได้เรียนรู้และทดลองในสมุดโน้ตเล่มนี้:
1. วิธีการใช้ Wrapper เพื่อประยุกต์ใช้ไลบรารี `pytorch-grad-cam` กับสถาปัตยกรรม YOLOv8
2. การรันการประมวลผลหาค่าความชันย้อนกลับ (Backpropagation) เจาะจงที่เป้าหมายวัตถุเฉพาะ
3. การแสดงผลเปรียบเทียบผลลัพธ์ของ **Grad-CAM** บน Backbone Layer (เลเยอร์ลึก) กับ Classification Head (เลเยอร์วิเคราะห์แยกชนิด)
4. การรันเปรียบเทียบกับวิธี **Eigen-CAM** (วิธีหาองค์ประกอบหลักแบบไม่อาศัยเกรเดียนต์)

---

## 🛠️ ขั้นตอนที่ 0: ตรวจสอบและนำเข้าไลบรารีที่จำเป็น

หากยังไม่ได้ติดตั้งไลบรารี กรุณาเปิด Terminal ใน virtual environment แล้วรัน:
```bash
pip install ultralytics grad-cam opencv-python matplotlib torch
```

In [ ]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

# นำเข้าไลบรารี XAI
try:
    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    print("[+] นำเข้าไลบรารีสำเร็จ พร้อมใช้งาน!")
except ImportError:
    print("[!] Error: ไม่พบไลบรารี pytorch-grad-cam")
    print("โปรดรัน: pip install grad-cam")

## 🧠 ขั้นตอนที่ 1: สร้าง Wrapper สำหรับโมเดล YOLO และกำหนด Box Target

เนื่องจาก YOLO คืนค่าผลลัพธ์ที่สลับซับซ้อน (ทั้งตำแหน่งกรอบและคะแนน) เราจึงสร้างคลาส wrapper `YOLOv8XAIWrapper` เพื่อห่อหุ้มเฉพาะการรันโมเดลหลัก และสร้างคลาส `YOLOBoxTarget` สำหรับชี้วัดดัชนีกล่องและประเภทวัตถุที่ใช้คำนวณ Gradient

In [ ]:
class YOLOv8XAIWrapper(torch.nn.Module):
    def __init__(self, yolomodel):
        super().__init__()
        self.model = yolomodel.model

    def forward(self, x):
        # คืนค่าผลลัพธ์ของการรันโมเดล PyTorch ดั้งเดิม
        return self.model(x)

class YOLOBoxTarget:
    def __init__(self, target_class_id, target_box_idx):
        self.target_class_id = target_class_id
        self.target_box_idx = target_box_idx

    def __call__(self, outputs):
        predictions = outputs[0]
        # ดึงคะแนนวิเคราะห์คลาสของกล่องวัตถุเป้าหมาย
        class_score = predictions[0, 4 + self.target_class_id, self.target_box_idx]
        return class_score

## 🖼️ ขั้นตอนที่ 2: โหลดและเตรียมรูปภาพนำเข้า

เราจะดึงรูปภาพ `seaweed_insect_detection.png` ซึ่งอยู่ในโฟลเดอร์ `images` นอกไดเรกทอรีนี้มาใช้ในการวิเคราะห์

In [ ]:
img_path = "../images/seaweed_insect_detection.png"

if not os.path.exists(img_path):
    print("[!] ไม่พบภาพตัวอย่าง กำลังสร้างรูปภาพจำลอง...")
    # สร้างรูปภาพจำลองวงกลมแดง
    img = np.zeros((640, 640, 3), dtype=np.uint8)
    cv2.circle(img, (320, 320), 100, (0, 0, 255), -1)
    cv2.putText(img, "Sample Target", (220, 220), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
else:
    print(f"[+] โหลดภาพสำเร็จจาก: {img_path}")
    img = cv2.imread(img_path)

# ปรับขนาดให้สอดคล้องกับ Input Size ของ YOLO (640x640)
img_resized = cv2.resize(img, (640, 640))
rgb_img = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
img_normalized = np.float32(rgb_img) / 255.0

# แปลงเป็น Tensor เพื่อเข้าสู่โครงข่ายประสาทเทียม
input_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).unsqueeze(0).float() / 255.0

# แสดงภาพเบื้องต้น
plt.figure(figsize=(6, 6))
plt.imshow(rgb_img)
plt.title("Input Image")
plt.axis("off")
plt.show()

## 🤖 ขั้นตอนที่ 3: รันโมเดล YOLOv8 และการเลือกเป้าหมายที่จะวิเคราะห์

เราโหลดโมเดลรุ่นน้ำหนักเบา `yolov8n.pt` และรันผลการประมวลภาพ จากนั้นเลือกกล่องทำนายมาทำ XAI

In [ ]:
print("-> กำลังโหลดน้ำหนักโมเดล YOLOv8...")
model = YOLO("yolov8n.pt")

# ทำนายภาพ
results = model(img_resized)
boxes = results[0].boxes

if len(boxes) == 0:
    print("[!] ไม่พบวัตถุจากการวิเคราะห์ จะรันโดยอ้างอิงตำแหน่งจำลอง")
    target_class_id = 0
    target_box_idx = 0
else:
    # แสดงรายการกล่องที่เจอ
    for idx, box in enumerate(boxes):
        cls_id = int(box.cls[0].item())
        conf = box.conf[0].item()
        print(f"ชิ้นที่ {idx}: คลาส {model.names[cls_id]} (Confidence: {conf:.2%})")
    
    # เลือกวัตถุแรกสุด
    target_class_id = int(boxes[0].cls[0].item())
    target_box_idx = 0
    print(f"-> คัดเลือกวิเคราะห์ชิ้นที่ {target_box_idx} (คลาส {model.names[target_class_id]})")

## 🔍 ขั้นตอนที่ 4: คำนวณหาค่า Grad-CAM และ Eigen-CAM เพื่อเปรียบเทียบผลลัพธ์

เราเลือกวิเคราะห์ที่เลเยอร์เป้าหมาย 2 เลเยอร์ ได้แก่:
1. **Backbone Layer 9:** เลเยอร์สกัดฟีเจอร์ระดับลึกที่สุดก่อนส่งเข้าหัวทำนาย (มักจะเห็นภาพรวมลักษณะโครงสร้างกายภาพใหญ่ๆ)
2. **Classification Head Layer:** เลเยอร์ตัดสินคลาสทำนาย (มักจะเห็นจุดจำเพาะที่เป็นเอกลักษณ์แยกสายพันธุ์วัตถุ)

In [ ]:
# เตรียม Wrapped Model
wrapped_model = YOLOv8XAIWrapper(model)
wrapped_model.eval()

# อ้างอิงเลเยอร์ใน YOLOv8-nano
backbone_layer = model.model.model[9] # Backbone P5
head_layer = model.model.model[22].cv3[0] # Classification head conv

# กำหนดเป้าหมาย
targets = [YOLOBoxTarget(target_class_id=target_class_id, target_box_idx=target_box_idx)]

# 1. คำนวณ Grad-CAM บน Backbone เลเยอร์ที่ 9
cam_backbone = GradCAM(model=wrapped_model, target_layers=[backbone_layer])
grayscale_cam_backbone = cam_backbone(input_tensor=input_tensor, targets=targets)[0, :]
vis_backbone = show_cam_on_image(img_normalized, grayscale_cam_backbone, use_rgb=True)

# 2. คำนวณ Grad-CAM บน Classification Head
cam_head = GradCAM(model=wrapped_model, target_layers=[head_layer])
grayscale_cam_head = cam_head(input_tensor=input_tensor, targets=targets)[0, :]
vis_head = show_cam_on_image(img_normalized, grayscale_cam_head, use_rgb=True)

# 3. คำนวณ Eigen-CAM บน Backbone
eigencam = EigenCAM(model=wrapped_model, target_layers=[backbone_layer])
grayscale_eigen = eigencam(input_tensor=input_tensor, targets=targets)[0, :]
vis_eigen = show_cam_on_image(img_normalized, grayscale_eigen, use_rgb=True)

print("[+] การประมวลผล XAI เสร็จสมบูรณ์!")

## 📊 ขั้นตอนที่ 5: พล็อตเปรียบเทียบข้อมูลแบบเปรียบเทียบ (Visualizations Comparison)

มาแสดงผลการวิเคราะห์เปรียบเทียบกัน!

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(rgb_img)
axes[0, 0].set_title("1. Original Resized Image")
axes[0, 0].axis("off")

axes[0, 1].imshow(vis_backbone)
axes[0, 1].set_title("2. Grad-CAM on Backbone (Layer 9)")
axes[0, 1].axis("off")

axes[1, 0].imshow(vis_head)
axes[1, 0].set_title("3. Grad-CAM on Classification Head")
axes[1, 0].axis("off")

axes[1, 1].imshow(vis_eigen)
axes[1, 1].set_title("4. Eigen-CAM on Backbone (Class-Agnostic)")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 📝 สรุปความเห็นส่วนการวิจัย

1. **เลเยอร์ลึกของ Backbone (Layer 9)** แสดงความร้อนที่ฟุ้งกว่า สกัดฟีเจอร์รูปร่างที่สำคัญในสเกลที่ใหญ่
2. **เลเยอร์ Head (Layer 22)** ตอบสนองแบบเจาะจงลงไปในบริเวณที่เป็นลักษณะเด่นที่ชี้ชี้ว่าคลาสของวัตถุนั้นคืออะไร ซึ่งพิสูจน์ให้เห็นถึงความพยายามของแบบจำลองในการมองหาสัญลักษณ์คลาสอย่างสมบูรณ์แบบ
3. **Eigen-CAM** จับโครงสร้างขอบและสัดส่วนจริงของวัตถุได้ชัดเจน คมชัด และไม่ขึ้นกับการจำแนกประเภท (Class-Agnostic)